PHASE 7: ANOMALY DETECTION(DBSCAN)

Uses density-based clustering to automatically detect anomalous laps per circuit, validating and enriching the manual cleaning rule used in Phase 2

In [1]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import joblib

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

SETUP

In [2]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'), exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase7_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

CONFIGURATION

In [3]:
DBSCAN_EPS = 0.6    #Neighborhood radius (in scaled feature units)
DBSCAN_MIN_SAMPLES = 5  #Minimum points to form a dense region
FEATURES = ['TyreLife', 'LapTimeSeconds', 'TrackTemp']
MIN_LAPS_FOR_CLUSTERING = 20    #Skip circuits with too little data

LOADING ENGINEERING DATA

We use the already CLEANED Phase 2 output as input here - DBSCAN is meant to catch anomalies that survival manual cleaning, not to re-clean the raw data from scratch

In [4]:
def load_data():
    path = os.path.join(BASE, 'data', 'processed', 'engineered_laps.csv')
    df = pd.read_csv(path)
    log.info(f"Loaded engineered_laps.csv: {len(df)} rows")

    df = df.dropna(subset=FEATURES)
    log.info(f"After dropping missing features values: {len(df)} rows")

    return df

RUN DBSCAN PER CIRCUIT

Fits DBSCAN separately for each circuit, since lap time scales and natural variance differ significantly between tracks - a single global model would treat Monaco and Monza laps on the same distance scale, which doesn't make sense

In [5]:
def run_dbscan_per_circuit(df):
    log.info("RUNNING DBSCAN ANOMALY DETECTION")
    log.info("-" * 50)

    all_results = []
    summary_rows = []
    dbscan_models = {}

    for circuit, group in df.groupby('CircuitName'):
        if len(group) < MIN_LAPS_FOR_CLUSTERING:
            log.warning(f" Skipping {circuit} - only {len(group)} laps")
            continue

        X = group[FEATURES].copy()

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        dbscan = DBSCAN(eps = DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
        labels = dbscan.fit_predict(X_scaled)

        '''DBSCAN labels noise points as -1; everything else
        is a cluster ID (0, 1, 2, ...). We only want the noise/not-noise
        distinction from anomaly detection'''
        group = group.copy()
        group['DBSCAN_Cluster'] = labels
        group['IsAnomaly'] = labels == -1

        n_anomalies = group['IsAnomaly'].sum()
        anomaly_rate = n_anomalies / len(group)

        log.info(f" {circuit}: {len(group)} laps |"
                f" {n_anomalies} anomalies ({anomaly_rate:.1%})")

        summary_rows.append({
            'CircuitName': circuit,
            'TotalLaps': len(group),
            'AnomaliesDetected': int(n_anomalies),
            'AnomalyRate': round(anomaly_rate, 4)
        })

        all_results.append(group)

        dbscan_models[circuit] = {'model': dbscan, 'scaler': scaler}

    results_df = pd.concat(all_results, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows)

    return results_df, summary_df, dbscan_models

CROSS-CHECK AGAINST WHAT PHASE 2 ALREADY EXCLUDED

Phase 2's raw_laps.csv (before cleaning) has many more laps than engineered_laps.csv (after cleaning). This function compares how many laps were removed by manual rules vs how many DBSCAN flags as anomalous WITHIN the already-cleaned set - telling us whether rules were thorough, and whether DBSCAN catches meaningful additional cases.

In [6]:
def compare_with_manual_cleaning(summary_df):
    log.info("COMPARING DBSCAN VS MANUAL CLEANING (PHASE 2)")
    log.info("-" * 50)

    raw_path = os.path.join(BASE, 'data', 'raw', 'raw_laps.csv')
    raw_laps = pd.read_csv(raw_path)

    raw_counts = raw_laps.groupby('CircuitName').size().rename('RawLapCount')
    comparison = summary_df.set_index('CircuitName').join(raw_counts)
    comparison['RemovedByManualCleaning'] = (
        comparison['RawLapCount'] - comparison['TotalLaps']
    )
    comparison['ManualCleaningRate'] = (
        comparison['RemovedByManualCleaning'] / comparison['RawLapCount']
    ).round(4)

    comparison = comparison.reset_index()

    log.info("\nManual cleaning rate vs DBSCAN anomaly rate (on cleaned data):")
    log.info(comparison[
        ['CircuitName', 'ManualCleaningRate', 'AnomalyRate']
    ].to_string(index=False))

    return comparison

MAIN PIPELINE

In [7]:
def main():
    log.info(" BOX-BOX PHASE 7: Anomaly Detection (DBSCAN)")
    log.info("-" * 50)

    df = load_data()

    results_df, summary_df, dbscan_models = run_dbscan_per_circuit(df)
    comparison_df = compare_with_manual_cleaning(summary_df)

    #Saving outputs
    anomaly_flags_path = os.path.join(
        BASE, 'data', 'processed', 'lap_anomaly_flags.csv'
    )

    #Save only identifying columns + flag, not the full dataset again
    flags_only = results_df[
        ['CircuitName', 'Driver', 'Stint', 'LapNumber', 'IsAnomaly', 'DBSCAN_Cluster']
    ]
    flags_only.to_csv(anomaly_flags_path, index=False)
    log.info(f"\n{anomaly_flags_path} with {len(flags_only)} rows")

    summary_path = os.path.join(
        BASE, 'data', 'outputs', 'dbscan_anomaly_summary.csv'
    )

    comparison_path = os.path.join(
        BASE, 'data', 'outputs', 'dbscan_anomaly_summary.csv'
    )

    comparison_df.to_csv(summary_path, index=False)
    log.info(f" {summary_path} with {len(comparison_df)} rows")

    models_path = os.path.join(
        BASE, 'models', 'dbscan_anomaly_models.pkl'
    )
    joblib.dump(dbscan_models, models_path)
    log.info(f"{models_path} with {len(dbscan_models)} circuit models")

    #Summary
    log.info(f"\nSUMMARY")
    log.info("-" * 50)

    total_laps = len(results_df)
    total_anomalies = results_df['IsAnomaly'].sum()
    log.info(f" Total laps analysed: {total_laps}")
    log.info(f" Total anomalies detected: {total_anomalies} "
            f"{total_anomalies/total_laps:.2f}")
    log.info(f"Top 5 circuits by DBSCAN anomaly rate:")
    top5 = comparison_df.sort_values('AnomalyRate', ascending = False).head(5)
    log.info(top5[['CircuitName', 'AnomalyRate']].to_string(index=False))

if __name__ == '__main__':
    main()

2026-07-28 07:08:46,511 - INFO -  BOX-BOX PHASE 7: Anomaly Detection (DBSCAN)
2026-07-28 07:08:46,513 - INFO - --------------------------------------------------
2026-07-28 07:08:46,613 - INFO - Loaded engineered_laps.csv: 20887 rows
2026-07-28 07:08:46,617 - INFO - After dropping missing features values: 20887 rows
2026-07-28 07:08:46,618 - INFO - RUNNING DBSCAN ANOMALY DETECTION
2026-07-28 07:08:46,618 - INFO - --------------------------------------------------
2026-07-28 07:08:46,629 - INFO -  Abu Dhabi: 868 laps | 8 anomalies (0.9%)
2026-07-28 07:08:46,639 - INFO -  Australia: 830 laps | 23 anomalies (2.8%)
2026-07-28 07:08:46,651 - INFO -  Austria: 1225 laps | 12 anomalies (1.0%)
2026-07-28 07:08:46,659 - INFO -  Azerbaijan: 826 laps | 21 anomalies (2.5%)
2026-07-28 07:08:46,669 - INFO -  Bahrain: 989 laps | 5 anomalies (0.5%)
2026-07-28 07:08:46,677 - INFO -  Belgium: 723 laps | 7 anomalies (1.0%)
2026-07-28 07:08:46,686 - INFO -  Canada: 249 laps | 3 anomalies (1.2%)
2026-07-28 